<a href="https://colab.research.google.com/github/Vivekpradhan13/Federated-learning-for-secure-medical-daignosis/blob/main/Federated_learning_for_secure_medical_diagnosis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import OrderedDict
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import CIFAR10
from torchvision.transforms import Compose, ToTensor, Normalize

# 1. THE DATA: Robust Splitting for 3 Virtual Hospitals
print("Loading data...")
transform = Compose([ToTensor(), Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
trainset = CIFAR10("./data", train=True, download=True, transform=transform)
testset = CIFAR10("./data", train=False, download=True, transform=transform)

num_clients = 3
# Robust math to ensure all images are accounted for
partition_size = len(trainset) // num_clients
remainders = len(trainset) % num_clients
lengths = [partition_size] * num_clients
lengths[0] += remainders # Put any leftover images in the first hospital

datasets = random_split(trainset, lengths, torch.Generator().manual_seed(42))
trainloaders = [DataLoader(ds, batch_size=32, shuffle=True) for ds in datasets]
testloader = DataLoader(testset, batch_size=32)

# 2. THE BRAIN: Neural Network Architecture
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

# 3. HELPER FUNCTIONS: Train, Test, and Average
def train(net, trainloader):
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.SGD(net.parameters(), lr=0.001, momentum=0.9)
    for images, labels in trainloader:
        optimizer.zero_grad()
        criterion(net(images), labels).backward()
        optimizer.step()

def test(net, testloader):
    criterion = torch.nn.CrossEntropyLoss()
    correct, total_loss = 0, 0.0
    with torch.no_grad():
        for images, labels in testloader:
            outputs = net(images)
            total_loss += criterion(outputs, labels).item()
            correct += (torch.max(outputs.data, 1)[1] == labels).sum().item()
    return total_loss / len(testloader), correct / len(testloader.dataset)

def average_weights(all_client_weights):
    new_weights = []
    for weights_list in zip(*all_client_weights):
        new_weights.append(np.array(weights_list).mean(axis=0))
    return new_weights

# 4. EXECUTION: The Federated Loop
print("\n--- Starting Federated Training ---")
global_weights = [val.cpu().numpy() for _, val in Net().state_dict().items()]

for round_num in range(1, 4):
    print(f"Round {round_num}")
    local_weights_list = []

    for i in range(num_clients):
        local_model = Net()
        # Load Global Weights
        params_dict = zip(local_model.state_dict().keys(), global_weights)
        local_model.load_state_dict(OrderedDict({k: torch.tensor(v) for k, v in params_dict}))

        # Local Training at "Hospital"
        train(local_model, trainloaders[i])

        local_weights_list.append([val.cpu().numpy() for _, val in local_model.state_dict().items()])

    # Server averages the "Notes"
    global_weights = average_weights(local_weights_list)

    # Accuracy check
    test_model = Net()
    params_dict = zip(test_model.state_dict().keys(), global_weights)
    test_model.load_state_dict(OrderedDict({k: torch.tensor(v) for k, v in params_dict}))
    _, accuracy = test(test_model, testloader)
    print(f" -> Global Accuracy: {accuracy:.2%}")

print("\nSuccess! Phase 1 is officially complete.")

Loading data...

--- Starting Federated Training ---
Round 1
 -> Global Accuracy: 17.01%
Round 2
 -> Global Accuracy: 22.05%
Round 3
 -> Global Accuracy: 30.27%

Success! Phase 1 is officially complete.


In [3]:
def secure_average_weights(all_client_weights, noise_multiplier=0.01):
    """
    Adds a layer of security by injecting 'noise' into the aggregated weights.
    This is a basic form of Differential Privacy.
    """
    new_weights = []
    for weights_list in zip(*all_client_weights):
        # 1. Calculate the standard average
        mean_weight = np.array(weights_list).mean(axis=0)

        # 2. Generate random 'noise' (Gaussian/Normal distribution)
        noise = np.random.normal(0, noise_multiplier, size=mean_weight.shape)

        # 3. Add the noise to the average to protect privacy
        secure_weight = mean_weight + noise
        new_weights.append(secure_weight)

    return new_weights

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [5]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, random_split

# The path from your successful kagglehub download
data_path = "/kaggle/input/chest-xray-pneumonia/chest_xray/train"

def load_medical_data(num_hospitals=3):
    # Medical images need specific resizing and normalization
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    full_dataset = torchvision.datasets.ImageFolder(root=data_path, transform=transform)

    # Robust math to avoid that 'ValueError' from your screenshot
    total_size = len(full_dataset)
    split_size = total_size // num_hospitals
    lengths = [split_size] * num_hospitals
    lengths[0] += (total_size % num_hospitals) # Add remainder to first hospital

    hospital_datasets = random_split(full_dataset, lengths)
    return [DataLoader(ds, batch_size=32, shuffle=True) for ds in hospital_datasets]

# Initialize our 3 Virtual Hospitals with X-Ray data
medical_trainloaders = load_medical_data(3)
print(f"Medical data loaded! Hospital 0 has {len(medical_trainloaders[0].dataset)} X-rays.")

Medical data loaded! Hospital 0 has 1740 X-rays.


In [6]:
class MedicalNet(nn.Module):
    def __init__(self):
        super(MedicalNet, self).__init__()
        # A slightly deeper model for complex X-ray patterns
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        # 224x224 images reduced twice by pooling (224 -> 112 -> 56)
        self.fc1 = nn.Linear(64 * 56 * 56, 128)
        self.fc2 = nn.Linear(128, 2) # ONLY 2 OUTPUTS: Normal or Pneumonia

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 56 * 56)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from collections import OrderedDict

# 1. SETUP MEDICAL DATA (Points to your Kagglehub path)
data_path = "/kaggle/input/chest-xray-pneumonia/chest_xray/train" # Verified from your screenshot

def prepare_medical_loaders(num_hospitals=3):
    transform = transforms.Compose([
        transforms.Resize((224, 224)), # X-rays are larger than CIFAR
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # Load from the folder you downloaded
    full_dataset = torchvision.datasets.ImageFolder(root=data_path, transform=transform)

    # Robust splitting to prevent 'ValueError'
    total_size = len(full_dataset)
    split_size = total_size // num_hospitals
    lengths = [split_size] * num_hospitals
    lengths[0] += (total_size % num_hospitals)

    datasets = random_split(full_dataset, lengths, torch.Generator().manual_seed(42))
    return [DataLoader(ds, batch_size=16, shuffle=True) for ds in datasets]

# 2. THE MEDICAL BRAIN (Binary Classification)
class MedicalNet(nn.Module):
    def __init__(self):
        super(MedicalNet, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        # After two 2x2 pools, 224x224 becomes 56x56
        self.fc1 = nn.Linear(32 * 56 * 56, 64)
        self.fc2 = nn.Linear(64, 2) # Output: 0 for Normal, 1 for Pneumonia

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 56 * 56)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# 3. SECURE AGGREGATION (Differential Privacy)
def secure_average_weights(all_weights, noise_level=0.01):
    """Adds mathematical noise to protect patient privacy."""
    new_weights = []
    for weights_list in zip(*all_weights):
        mean_w = np.array(weights_list).mean(axis=0)
        noise = np.random.normal(0, noise_level, size=mean_w.shape)
        new_weights.append(mean_w + noise)
    return new_weights

# 4. TRAINING ENGINE
def train_hospital(model, loader):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)
    criterion = nn.CrossEntropyLoss()
    # Training on just 20 batches per round for speed in Colab
    for i, (images, labels) in enumerate(loader):
        if i > 20: break
        optimizer.zero_grad()
        criterion(model(images), labels).backward()
        optimizer.step()

# 5. THE FINAL EXECUTION LOOP
print("--- Loading Medical X-Rays ---")
medical_loaders = prepare_medical_loaders(3)
global_weights = [val.cpu().numpy() for _, val in MedicalNet().state_dict().items()]

print("\n--- Starting Secure Federated Diagnosis ---")
for round_num in range(1, 4):
    print(f"Round {round_num}: Collaborative Learning in progress...")
    local_weights_list = []

    for h_id in range(3):
        local_model = MedicalNet()
        # Load global knowledge
        params = zip(local_model.state_dict().keys(), global_weights)
        local_model.load_state_dict(OrderedDict({k: torch.tensor(v) for k, v in params}))

        # Local hospital training
        train_hospital(local_model, medical_loaders[h_id])

        # Send 'Secure' updates back
        local_weights_list.append([val.cpu().numpy() for _, val in local_model.state_dict().items()])

    # Server merges knowledge with Privacy Noise
    global_weights = secure_average_weights(local_weights_list)
    print(f" Round {round_num} complete. Global Model updated securely.")

print("\n🎉 PROJECT COMPLETE!")

--- Loading Medical X-Rays ---

--- Starting Secure Federated Diagnosis ---
Round 1: Collaborative Learning in progress...
 Round 1 complete. Global Model updated securely.
Round 2: Collaborative Learning in progress...
 Round 2 complete. Global Model updated securely.
Round 3: Collaborative Learning in progress...
 Round 3 complete. Global Model updated securely.

🎉 PROJECT COMPLETE!
